# Phase 3 — Multilingual document extraction

This notebook processes the pinned immutable Phase 2 snapshot. It uses digital
text first and OCR only when required. It is resumable and performs no LLM calls.

Run **Runtime → Run all**. Do not unzip the package manually.

## 1. Fast setup — mount the known project directly

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

EXPECTED_PACKAGE_SHA256 = "37ddb9172afaf05f9481941fc05311f4806d5eadeb8cb9f1ad145588c444fd28"
PACKAGE_FILENAME = "PHASE_3_MULTILINGUAL_EXTRACTION_PACKAGE.zip"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Devoteam internship/Devoteam_AI_CLEAN_PIPELINE")
assert (PROJECT_ROOT / "config" / "project.yaml").exists(), f"Clean project not found: {PROJECT_ROOT}"
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
print(f"Project root: {PROJECT_ROOT}")

## 2. Verify and install the signed Phase 3 package

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_package_sha = file_sha256(PACKAGE_PATH)
assert actual_package_sha == EXPECTED_PACKAGE_SHA256, "Phase 3 package hash mismatch"

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        assert str(target).startswith(str(PROJECT_ROOT.resolve()) + os.sep), f"Unsafe archive path: {member.filename}"
    package_manifest = json.loads(archive.read("PHASE_3_PACKAGE_MANIFEST.json"))
    for entry in package_manifest["files"]:
        assert hashlib.sha256(archive.read(entry["path"])).hexdigest() == entry["sha256"]
    for member in archive.infolist():
        if member.is_dir() or member.filename == "PHASE_3_PACKAGE_MANIFEST.json":
            continue
        destination = PROJECT_ROOT / member.filename
        packaged_hash = hashlib.sha256(archive.read(member.filename)).hexdigest()
        if destination.exists():
            assert file_sha256(destination) == packaged_hash, f"Refusing to overwrite changed file: {member.filename}"
        else:
            archive.extract(member, PROJECT_ROOT)

print(f"Verified package SHA-256: {actual_package_sha}")
print("Phase 3 extension installed safely.")

## 3. Install OCR/Python requirements once and run tests

In [ ]:
language_result = subprocess.run(
    ["tesseract", "--list-langs"], text=True, capture_output=True
) if shutil.which("tesseract") else None
installed_languages = set(language_result.stdout.splitlines()[1:]) if language_result and language_result.returncode == 0 else set()
missing_languages = {"eng", "fra", "ara"} - installed_languages
if missing_languages:
    print(f"Installing OCR language packs {sorted(missing_languages)} — usually 1–3 minutes...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "tesseract-ocr", "tesseract-ocr-fra", "tesseract-ocr-eng", "tesseract-ocr-ara"],
        check=True,
    )
else:
    print("Tesseract French/English/Arabic language packs already available.")

print("Installing/checking Python packages — usually 1–3 minutes...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements" / "phase3.txt")],
    check=True,
)
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
assert tests.returncode == 0, "Tests failed; extraction did not start."
print("All foundation, snapshot, and extraction tests passed.")

## 4. Verify the immutable input and run/resume extraction

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from devoteam_reference_ai.phase3_pipeline import run_phase3

summary = run_phase3(
    project_root=PROJECT_ROOT,
    config_path=PROJECT_ROOT / "config" / "phase3_extraction.yaml",
    progress=print,
)
print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))

## 5. Verify output hashes and publish the Phase 3 gate

In [ ]:
from devoteam_reference_ai.phase3_pipeline import verify_phase3

assert summary["status"] == "PASS", "One or more documents failed. Rerun the notebook to retry only failed documents."
verified = verify_phase3(Path(summary["run_root"]))
assert verified["status"] == "PASS", f"Phase 3 has processing failures: {verified}"
assert verified["documents_completed"] == verified["documents_expected"] == 134
assert verified["source_snapshot_mutation_calls"] == 0
assert verified["external_llm_calls"] == 0

print("PHASE 3: PASS")
print(f"Pages extracted: {verified['pages_total']}")
print(f"Digital pages: {verified['digital_pages']}")
print(f"OCR pages: {verified['ocr_pages']}")
print(f"QA gate: {verified['qa_gate']}")
print(f"Review pages: {verified['review_pages']}")
print(f"Failed pages: {verified['failed_pages']}")
print("Send this final block for senior review before Phase 4.")